# 05 — Out-of-sample evaluation: every model on the same footing

**Reads:** `data/raw/prices_yahoo.csv`, `data/processed/calendar.csv`, `data/processed/pred_<model>.csv`.

**Writes** (all in `results/tables/`): `predictive.csv`, `investment.csv`, `ic_by_period.csv`,
`period_returns.csv`, `daily_returns.csv`, `random_portfolios.csv`, `random_summary.csv`, `holdings.csv`,
`difficulty.csv`, `eval_settings.json`.

The class script scores its own backtest with formulas that do not line up (notebook 03). Here every model and
every benchmark goes through one procedure:

1. On each rebalance date (first trading day of the month) the model ranks the 96 stocks.
2. The ten highest-ranked stocks are bought in equal amounts at the **next** day's close and held to the next
   month's entry date. Daily returns come from daily prices, so every line covers exactly the same days.
3. **Predictive performance** asks whether the ranking was right: did the stocks ranked higher go on to earn
   more over the following month?
4. **Investment performance** asks whether the portfolio made money, against three yardsticks: SPY, an
   equal-weighted portfolio of all 96 stocks, and 10,000 portfolios that pick ten of the 96 at random each month.

The equal-weighted universe and the random portfolios matter because the 96 tickers were chosen in 2026: they
are companies known to be large today, so a portfolio drawn from them starts ahead of the index whatever the
model does.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"

SEED = 6400
TOP_N = 10
N_RANDOM = 10_000
COST_BPS = 10                      # cost per dollar bought or sold, in basis points (1 bp = 0.01%)
TRADING_DAYS = 252

MODELS = {                         # file suffix -> label used in the tables and figures
    "original": "Class model",
    "clean": "Step 1: new features and target",
    "long": "Step 2: 13 years of training data",
    "improved": "Step 3: tuned and refitted monthly (improved model)",
    "linear": "Linear regression, same features",
    "momentum": "12-month momentum rule",
}
RETURN_FORECASTS = ["original", "clean", "long", "improved", "linear"]   # models whose score is a return forecast

## Prices, the calendar and each stock's return over each holding period

In [2]:
px = pd.read_csv(RAW / "prices_yahoo.csv", parse_dates=["date"])
adj_all = px[px["symbol"] != "^IRX"].pivot(index="date", columns="symbol", values="adjusted")
ret_all = adj_all.pct_change(fill_method=None)            # includes SPY as a column
STOCKS = [c for c in adj_all.columns if c != "SPY"]
# 13-week Treasury-bill yield, % a year -> a daily risk-free return
rf_daily = (px[px["symbol"] == "^IRX"].set_index("date")["close"] / 100 / TRADING_DAYS).reindex(adj_all.index).ffill()

calendar = pd.read_csv(PROC / "calendar.csv", parse_dates=["rebalance", "entry", "exit"])
periods = calendar.dropna(subset=["exit"]).reset_index(drop=True)      # complete holding periods only
first_day, last_day = periods["entry"].iloc[0], periods["exit"].iloc[-1]

# period_ret[p, i]: return of stock i from the entry close of period p to the next entry close
period_ret = pd.DataFrame(adj_all.loc[periods["exit"], STOCKS].to_numpy()
                          / adj_all.loc[periods["entry"], STOCKS].to_numpy() - 1,
                          index=periods["rebalance"], columns=STOCKS)
period_ret.round(6).to_csv(TAB / "period_returns.csv")
relative_ret = period_ret.sub(period_ret.mean(axis=1), axis=0)          # minus the average stock that period
years = (last_day - first_day).days / 365.25
print(f"{len(periods)} holding periods, {first_day.date()} to {last_day.date()} ({years:.2f} years), {len(STOCKS)} stocks")

34 holding periods, 2024-01-03 to 2026-11-03 (2.83 years), 96 stocks


## Scores: one row per rebalance date, one column per stock

In [3]:
scores = {}
for name in MODELS:
    p = pd.read_csv(PROC / f"pred_{name}.csv", parse_dates=["date"])
    wide = p[p["date"].isin(periods["rebalance"])].pivot(index="date", columns="symbol", values="pred")
    scores[name] = wide.reindex(index=periods["rebalance"], columns=STOCKS)
    assert scores[name].notna().sum(axis=1).min() >= TOP_N, name
{k: v.shape for k, v in scores.items()}

{'original': (34, 96),
 'clean': (34, 96),
 'long': (34, 96),
 'improved': (34, 96),
 'linear': (34, 96),
 'momentum': (34, 96)}

## Predictive performance

- **Rank IC** (information coefficient): each month, the rank correlation across the 96 stocks between the
  model's score and the return that followed. Zero means the ranking carried no information; quantitative
  investors treat a steady 0.02–0.05 as a useful signal. The t-statistic is the average monthly IC divided by
  its standard error over the months.
- **Out-of-sample R²**: one minus the model's squared forecast error over the squared error of forecasting
  "every stock earns the average". Positive means the model beat that naive forecast; negative means it did
  worse. Forecasts and outcomes are both measured relative to the average stock.
- **Top ten minus the average stock**: the mean monthly return of the model's ten picks over the mean of all 96.
- **Hit rate**: the share of picks that beat the median stock that month. A coin flip gives 50%.

The class model's last training rows have targets that end in late January 2024, inside the first test period.
The two "after first period" columns drop that period to check the result does not rest on it.

In [4]:
ic = pd.DataFrame({name: [stats.spearmanr(s.loc[d].dropna(), period_ret.loc[d, s.loc[d].dropna().index])[0]
                          for d in s.index] for name, s in scores.items()}, index=periods["rebalance"])
ic.round(4).to_csv(TAB / "ic_by_period.csv")

picks = {name: {d: s.loc[d].dropna().nlargest(TOP_N).index.tolist() for d in s.index} for name, s in scores.items()}
median_ret = period_ret.median(axis=1)

rows = []
for name, label in MODELS.items():
    s = scores[name]
    spread = pd.Series({d: period_ret.loc[d, picks[name][d]].mean() - period_ret.loc[d].mean() for d in s.index})
    hits = np.mean([(period_ret.loc[d, picks[name][d]] > median_ret[d]).mean() for d in s.index])
    if name in RETURN_FORECASTS:
        forecast = s.sub(s.mean(axis=1), axis=0)
        ok = forecast.notna()
        r2 = 1 - ((relative_ret - forecast)[ok] ** 2).sum().sum() / (relative_ret[ok] ** 2).sum().sum()
    else:
        r2 = np.nan
    rows.append(dict(model=name, label=label,
                     rank_ic_mean=ic[name].mean(), rank_ic_sd=ic[name].std(ddof=1),
                     rank_ic_t=ic[name].mean() / (ic[name].std(ddof=1) / np.sqrt(len(ic))),
                     share_months_ic_positive=(ic[name] > 0).mean(),
                     rank_ic_mean_after_first_period=ic[name].iloc[1:].mean(),
                     rank_ic_t_after_first_period=ic[name].iloc[1:].mean() / (ic[name].iloc[1:].std(ddof=1) / np.sqrt(len(ic) - 1)),
                     r2_oos=r2,
                     top10_minus_average_pp=100 * spread.mean(),
                     top10_minus_average_t=spread.mean() / (spread.std(ddof=1) / np.sqrt(len(spread))),
                     hit_rate=hits))
predictive = pd.DataFrame(rows).set_index("model")
predictive.round(4).to_csv(TAB / "predictive.csv")
predictive.drop(columns="label").round(3)

,rank_ic_mean,rank_ic_sd,rank_ic_t,share_months_ic_positive,rank_ic_mean_after_first_period,rank_ic_t_after_first_period,r2_oos,top10_minus_average_pp,top10_minus_average_t,hit_rate
model,,,,,,,,,,
original,0.045,0.125,2.107,0.706,0.036,1.803,-0.063,0.714,1.143,0.535
clean,0.012,0.135,0.529,0.559,0.011,0.446,-0.121,0.071,0.077,0.497
long,0.000,0.128,0.020,0.471,0.004,0.167,-0.012,1.163,1.118,0.515
improved,0.024,0.205,0.691,0.588,0.025,0.690,0.003,1.531,1.069,0.500
linear,0.029,0.220,0.767,0.559,0.030,0.779,0.001,0.898,0.654,0.515
momentum,0.041,0.262,0.917,0.588,0.024,0.559,NaN,1.409,1.018,0.526


## Investment performance: one backtest for every strategy

`backtest` takes the weights chosen on each rebalance date and returns the portfolio's daily returns. Within a
holding period the positions are left alone, so weights drift with prices. At each rebalance the cost is
`COST_BPS` on every dollar bought or sold. **Turnover** is the share of the portfolio replaced in an average
month.

In [5]:
def backtest(weights, cost_bps=0.0):
    """weights: {rebalance date: Series of weights summing to 1}. Returns (daily returns, mean monthly turnover)."""
    daily, turnover = [], []
    held = pd.Series(dtype=float)                       # weights at the end of the previous period
    for reb, entry, exit_ in periods[["rebalance", "entry", "exit"]].itertuples(index=False):
        w = weights[reb]
        window = ret_all.loc[(ret_all.index > entry) & (ret_all.index <= exit_), w.index]
        value = (1 + window).cumprod() * w.to_numpy()   # value of each position, starting from its weight
        total = value.sum(axis=1)
        r = total.pct_change()
        r.iloc[0] = total.iloc[0] - 1
        traded = w.sub(held, fill_value=0).abs().sum()  # dollars bought plus dollars sold, per dollar of portfolio
        r.iloc[0] = (1 + r.iloc[0]) * (1 - cost_bps / 1e4 * traded) - 1
        if len(held):
            turnover.append(traded / 2)
        held = value.iloc[-1] / total.iloc[-1]
        daily.append(r)
    return pd.concat(daily), float(np.mean(turnover))


def equal_weight(names):
    return pd.Series(1 / len(names), index=names)


def script_weights(score_row):
    """The class script's rule: top ten, weight proportional to the prediction, negatives set to zero."""
    top = score_row.dropna().nlargest(TOP_N).clip(lower=0)
    return top / top.sum() if top.sum() > 0 else equal_weight(top.index)


strategies = {
    "spy": ("SPY (S&P 500 ETF)", {d: pd.Series({"SPY": 1.0}) for d in periods["rebalance"]}),
    "universe": ("All 96 stocks, equal weight", {d: equal_weight(STOCKS) for d in periods["rebalance"]}),
}
for name, label in MODELS.items():
    strategies[name] = (label, {d: equal_weight(picks[name][d]) for d in periods["rebalance"]})
strategies["original_script_weights"] = ("Class model, the script's own weights",
                                         {d: script_weights(scores["original"].loc[d]) for d in periods["rebalance"]})

daily, daily_net, turnover = {}, {}, {}
for key, (label, w) in strategies.items():
    daily[key], turnover[key] = backtest(w)
    daily_net[key], _ = backtest(w, COST_BPS)
daily = pd.DataFrame(daily)
daily_net = pd.DataFrame(daily_net)
daily.round(8).to_csv(TAB / "daily_returns.csv", index_label="date")
daily_net.round(8).to_csv(TAB / "daily_returns_net.csv", index_label="date")

## The performance table

- **Sharpe ratio**: average daily return above the Treasury-bill rate, divided by its standard deviation,
  times √252 (the returns here are daily, so the daily factor is the right one).
- **Alpha and beta against SPY**: the intercept and slope from regressing the strategy's daily excess return on
  SPY's. Beta is how much market risk the portfolio carries; alpha is the return left after that.
- **Active return against the 96-stock portfolio**: the fair test of stock *selection*, since every strategy
  draws from the same list. The t-statistic says whether the average daily gap is distinguishable from zero.

In [6]:
def performance(r, spy_r, universe_r):
    n_years = len(r) / TRADING_DAYS
    growth = (1 + r).cumprod()
    excess = r - rf_daily.reindex(r.index)
    spy_excess = spy_r - rf_daily.reindex(r.index)
    fit = stats.linregress(spy_excess, excess)
    active = r - universe_r
    with np.errstate(divide="ignore", invalid="ignore"):      # SPY against itself has no residual
        alpha_t = fit.intercept / fit.intercept_stderr
    return dict(total_return=growth.iloc[-1] - 1,
                annual_return=growth.iloc[-1] ** (1 / n_years) - 1,
                annual_volatility=r.std(ddof=1) * np.sqrt(TRADING_DAYS),
                sharpe=excess.mean() / excess.std(ddof=1) * np.sqrt(TRADING_DAYS),
                max_drawdown=(growth / growth.cummax() - 1).min(),
                beta=fit.slope,
                alpha_annual=fit.intercept * TRADING_DAYS,
                alpha_t=alpha_t,
                active_return_annual=active.mean() * TRADING_DAYS,
                tracking_error=active.std(ddof=1) * np.sqrt(TRADING_DAYS),
                active_t=(active.mean() / (active.std(ddof=1) / np.sqrt(len(active)))) if active.abs().sum() > 0 else np.nan)


rows = []
for key, (label, _) in strategies.items():
    row = dict(strategy=key, label=label, **performance(daily[key], daily["spy"], daily["universe"]))
    net = (1 + daily_net[key]).cumprod().iloc[-1]
    row.update(monthly_turnover=turnover[key], total_return_net=net - 1,
               annual_return_net=net ** (TRADING_DAYS / len(daily_net)) - 1)
    rows.append(row)
investment = pd.DataFrame(rows).set_index("strategy")
investment.loc["spy", ["alpha_annual", "alpha_t"]] = np.nan          # SPY has no alpha against itself
investment.round(4).to_csv(TAB / "investment.csv")
investment.drop(columns="label").round(3)

,total_return,annual_return,annual_volatility,sharpe,max_drawdown,beta,alpha_annual,alpha_t,active_return_annual,tracking_error,active_t,monthly_turnover,total_return_net,annual_return_net
strategy,,,,,,,,,,,,,,
spy,0.662,0.197,0.154,0.967,-0.188,1.000,NaN,NaN,-0.002,0.067,-0.045,0.000,0.660,0.197
universe,0.686,0.203,0.131,1.151,-0.165,0.769,0.036,1.083,0.000,0.000,NaN,0.031,0.680,0.202
original,1.094,0.299,0.198,1.209,-0.199,0.939,0.099,1.231,0.088,0.130,1.139,0.637,1.006,0.280
clean,0.640,0.192,0.238,0.677,-0.242,1.286,-0.031,-0.395,0.010,0.160,0.104,0.611,0.574,0.174
long,1.332,0.350,0.240,1.192,-0.187,1.284,0.095,1.169,0.135,0.166,1.365,0.567,1.244,0.332
improved,1.496,0.383,0.311,1.061,-0.259,1.627,0.087,0.796,0.179,0.245,1.226,0.237,1.455,0.375
linear,1.042,0.288,0.310,0.834,-0.289,1.653,0.011,0.110,0.107,0.241,0.747,0.361,0.992,0.277
momentum,1.424,0.369,0.289,1.082,-0.238,1.428,0.100,0.893,0.162,0.237,1.152,0.292,1.376,0.359
original_script_weights,1.507,0.385,0.218,1.409,-0.225,0.968,0.163,1.716,0.156,0.153,1.715,0.648,1.400,0.364


## Skill or the luck of the draw: 10,000 random portfolios

Each random portfolio follows the same calendar and the same rule as the models, except that its ten stocks are
drawn at random from the 96 every month. The spread of their results is what "no skill" looks like on this
list of stocks over these 34 months. A model's **percentile** is the share of random portfolios it beat.

In [7]:
rng = np.random.default_rng(SEED)
R = period_ret.to_numpy()
random_period = np.empty((N_RANDOM, len(periods)))
for p in range(len(periods)):
    draw = rng.random((N_RANDOM, len(STOCKS))).argpartition(TOP_N, axis=1)[:, :TOP_N]    # ten random columns per row
    random_period[:, p] = R[p][draw].mean(axis=1)
random_total = (1 + random_period).prod(axis=1) - 1
periods_per_year = len(periods) / years
random_ratio = random_period.mean(axis=1) / random_period.std(axis=1, ddof=1) * np.sqrt(periods_per_year)
pd.DataFrame({"total_return": random_total, "return_to_risk": random_ratio}).round(6).to_csv(
    TAB / "random_portfolios.csv", index_label="draw")

rows = []
for key, (label, w) in strategies.items():
    if key == "spy":
        own = adj_all.loc[periods["exit"], "SPY"].to_numpy() / adj_all.loc[periods["entry"], "SPY"].to_numpy() - 1
    else:
        own = np.array([(period_ret.loc[d, w[d].index] * w[d]).sum() for d in periods["rebalance"]])
    total = (1 + own).prod() - 1
    ratio = own.mean() / own.std(ddof=1) * np.sqrt(periods_per_year)
    rows.append(dict(strategy=key, label=label, total_return=total,
                     percentile_total_return=100 * (random_total < total).mean(),
                     share_random_at_least_as_good=(random_total >= total).mean(),
                     return_to_risk=ratio, percentile_return_to_risk=100 * (random_ratio < ratio).mean()))
random_summary = pd.DataFrame(rows).set_index("strategy")
random_summary.round(4).to_csv(TAB / "random_summary.csv")
print(f"random portfolios: median total return {np.median(random_total):.1%}, "
      f"5th–95th percentile {np.quantile(random_total, 0.05):.1%} to {np.quantile(random_total, 0.95):.1%}; "
      f"share beating SPY {(random_total > random_summary.loc['spy', 'total_return']).mean():.1%}")
random_summary.drop(columns="label").round(3)

random portfolios: median total return 66.3%, 5th–95th percentile 29.4% to 115.2%; share beating SPY 50.3%


,total_return,percentile_total_return,share_random_at_least_as_good,return_to_risk,percentile_return_to_risk
strategy,,,,,
spy,0.662,49.71,0.503,1.686,72.07
universe,0.686,53.49,0.465,1.982,89.54
original,1.094,92.97,0.070,1.729,75.16
clean,0.640,46.26,0.537,0.911,9.92
long,1.332,98.53,0.015,1.356,42.40
improved,1.496,99.45,0.006,1.167,26.13
linear,1.042,90.66,0.093,0.960,12.22
momentum,1.424,99.13,0.009,1.174,26.53
original_script_weights,1.507,99.47,0.005,1.719,74.44


## What each model held

Two models that hold the same stocks, or whose returns move together, are making the same bet under different
names. The comparison of interest is the improved model against the momentum rule, which uses no model at all.

In [8]:
holdings = pd.DataFrame([dict(model=name, rebalance=d.date().isoformat(), stocks=" ".join(sorted(picks[name][d])))
                         for name in MODELS for d in periods["rebalance"]])
holdings.to_csv(TAB / "holdings.csv", index=False)
most_held = {name: pd.Series([s for d in periods["rebalance"] for s in picks[name][d]]).value_counts().head(8)
             for name in MODELS}
most_held = pd.DataFrame({k: (v.index + " (" + v.astype(str).to_numpy() + ")").to_numpy() for k, v in most_held.items()})
most_held.to_csv(TAB / "most_held.csv", index=False)

# stocks two models hold in common in an average month, out of ten
overlap = pd.DataFrame({a: {b: np.mean([len(set(picks[a][d]) & set(picks[b][d])) for d in periods["rebalance"]])
                            for b in MODELS} for a in MODELS})
overlap.round(2).to_csv(TAB / "holdings_overlap.csv")
# correlation of daily returns in excess of the 96-stock portfolio
active_corr = daily[list(MODELS)].sub(daily["universe"], axis=0).corr()
active_corr.round(3).to_csv(TAB / "active_return_correlation.csv")
print("stocks held in common in an average month (of 10):")
print(overlap.round(1).to_string())
print("\ncorrelation of daily active returns:")
print(active_corr.round(2).to_string())
print("\nmost-held stocks (months held, of 34):")
print(most_held.to_string(index=False))

stocks held in common in an average month (of 10):
          original  clean  long  improved  linear  momentum
original      10.0    2.6   3.2       2.7     3.1       1.8
clean          2.6   10.0   3.7       4.4     3.8       2.8
long           3.2    3.7  10.0       5.1     5.2       3.0
improved       2.7    4.4   5.1      10.0     6.3       4.9
linear         3.1    3.8   5.2       6.3    10.0       4.1
momentum       1.8    2.8   3.0       4.9     4.1      10.0

correlation of daily active returns:
          original  clean  long  improved  linear  momentum
original      1.00   0.44  0.40      0.32    0.29      0.16
clean         0.44   1.00  0.67      0.69    0.70      0.55
long          0.40   0.67  1.00      0.71    0.70      0.55
improved      0.32   0.69  0.71      1.00    0.90      0.82
linear        0.29   0.70  0.70      0.90    1.00      0.80
momentum      0.16   0.55  0.55      0.82    0.80      1.00

most-held stocks (months held, of 34):
  original     clean      long 

## Why this is hard, in three numbers

- how much one stock's monthly return differs from the average stock's (the noise);
- how much the improved model's forecasts differ from one stock to the next (the claimed signal);
- how many years of data it would take for a 2-point annual edge to stand out from zero at the usual
  threshold (t = 2), given how far a ten-stock portfolio strays from the 96-stock portfolio in a year
  (t = edge / tracking error × √years, so years = (2 × tracking error / edge)²); and the same for the edge the
  improved model actually showed.

In [9]:
noise = relative_ret.std(axis=1, ddof=1).mean()
signal = scores["improved"].sub(scores["improved"].mean(axis=1), axis=0).std(axis=1, ddof=1).mean()
te = investment.loc["improved", "tracking_error"]
difficulty = pd.Series({
    "monthly_relative_return_sd_pp": 100 * noise,
    "improved_forecast_sd_pp": 100 * signal,
    "noise_to_signal": noise / signal,
    "tracking_error_improved_pp": 100 * te,
    "years_for_2pt_edge_to_reach_t2": (2 * te / 0.02) ** 2,
    "improved_active_return_annual_pp": 100 * investment.loc["improved", "active_return_annual"],
    "years_for_observed_edge_to_reach_t2": (2 * te / investment.loc["improved", "active_return_annual"]) ** 2,
    "test_years": years,
})
difficulty.rename("value").round(3).to_csv(TAB / "difficulty.csv", index_label="statistic")
json.dump({"seed": SEED, "top_n": TOP_N, "n_random": N_RANDOM, "cost_bps": COST_BPS, "periods": len(periods),
           "first_day": first_day.date().isoformat(), "last_day": last_day.date().isoformat(),
           "stocks": len(STOCKS), "years": round(years, 3)}, open(TAB / "eval_settings.json", "w"), indent=2)
difficulty.round(2)

monthly_relative_return_sd_pp            8.57
improved_forecast_sd_pp                  0.34
noise_to_signal                         24.96
tracking_error_improved_pp              24.53
years_for_2pt_edge_to_reach_t2         601.89
improved_active_return_annual_pp        17.90
years_for_observed_edge_to_reach_t2      7.51
test_years                               2.83
dtype: float64